# 01 : Exploration des séries Eurostat

**Auteur :** Tahar Guenfoud | Mai 2026  
**Source :** API Eurostat JSON (ec.europa.eu) — stat.nbb.be est hors service depuis mars 2026

## Objectifs de ce notebook

1. Vérifier que `EurostatClient` récupère bien les 5 séries cibles
2. Assembler le DataFrame mensuel consolidé
3. Produire les premières visualisations Plotly interactives
4. Documenter la narrative bancaire : cycle BCE 2022-2024

## Séries utilisées

| Clé interne | Code Eurostat | Description | Pertinence bancaire |
|-------------|--------------|-------------|--------------------|
| `olo_10y` | `irt_lt_mcby_m` | Taux OLO belge 10 ans | Coût de financement long terme |
| `euribor_3m` | `irt_h_mr3_m` | EURIBOR 3M (zone euro) | Taux court terme BCE |
| `inflation` | `prc_hicp_manr` | Inflation IPCH annuelle | Pression sur le remboursement |
| `chomage` | `une_rt_m` | Taux de chômage mensuel | Solvabilité des emprunteurs |
| `pib` | `nama_10_gdp` | PIB Belgique | Santé macroéconomique globale |


---
## 1. Setup

In [ ]:
import sys
import importlib
from pathlib import Path

# Chercher la racine du projet en remontant jusqu'à trouver src/
# Fonctionne que Jupyter soit lancé depuis macrodash-nbb/ ou notebooks/
ROOT = Path().resolve()
for _ in range(4):
    if (ROOT / "src").exists():
        break
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Reload pour prendre les dernières modifications sans redémarrer le kernel
import src.config
import src.fetch_eurostat
import src.transform
importlib.reload(src.config)
importlib.reload(src.fetch_eurostat)
importlib.reload(src.transform)

import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from src.config import DATASETS, BCE_EVENTS, DATA_RAW, DATA_PROCESSED
from src.fetch_eurostat import EurostatClient
from src.transform import build_macro_df

print(f"ROOT        : {ROOT}")
print(f"DATA_RAW    : {DATA_RAW}")
print(f"Datasets    : {list(DATASETS.keys())}")
print("Imports OK")

---
## 2. Récupération des données

La première exécution appelle l'API Eurostat (environ 10 secondes).  
Les fois suivantes, le cache CSV dans `data/raw/` est utilisé directement.

In [ ]:
client = EurostatClient()

cached = EurostatClient.load_cached()
if len(cached) == len(DATASETS):
    print(f"Cache complet ({len(cached)} séries) : chargement depuis data/raw/")
    series = cached
else:
    print(f"Cache incomplet ({len(cached)}/{len(DATASETS)}) : appel API Eurostat")
    series = client.fetch_all(start="2010", cache=True)

print()
print("INVENTAIRE")
print("-" * 50)
for k, df in series.items():
    if not df.empty:
        print(f"  {k:<15} {len(df):>4} obs  |  {df['periode'].min()} -> {df['periode'].max()}")
    else:
        print(f"  {k:<15} VIDE")

---
## 3. Construction du DataFrame mensuel

In [ ]:
macro = build_macro_df(series)

if macro.empty:
    print("DataFrame vide : relancer la cellule 2.")
else:
    print(f"Shape   : {macro.shape}")
    print(f"Période : {macro.index.min().date()} -> {macro.index.max().date()}")
    print(f"NaN par colonne :")
    print(macro.isna().sum())
    display(macro.tail(6))

---
## 4. Visualisation Plotly : séries individuelles

On trace les 4 séries principales en subplots avec axe X partagé.  
Les graphiques Plotly sont interactifs : zoom, survol, export PNG.

In [ ]:
SERIES_TO_PLOT = [k for k in ("olo_10y", "euribor_3m", "inflation", "chomage") if k in macro.columns]

if not SERIES_TO_PLOT:
    print("Pas de données disponibles. Relancer les cellules 2 et 3 d'abord.")
else:
    fig = make_subplots(
        rows=len(SERIES_TO_PLOT),
        cols=1,
        shared_xaxes=True,
        subplot_titles=[DATASETS[k]["label"] for k in SERIES_TO_PLOT],
        vertical_spacing=0.06,
    )

    for i, key in enumerate(SERIES_TO_PLOT, start=1):
        cfg = DATASETS[key]
        s = macro[key].dropna()
        fig.add_trace(
            go.Scatter(
                x=s.index,
                y=s.values,
                mode="lines",
                name=cfg["label"],
                line=dict(color=cfg["color"], width=2),
                hovertemplate="%{x|%Y-%m}: %{y:.2f} " + cfg["unit"] + "<extra></extra>",
            ),
            row=i,
            col=1,
        )

    fig.update_layout(
        height=280 * len(SERIES_TO_PLOT),
        title_text="Indicateurs macroéconomiques belges 2010-2026 (Eurostat)",
        title_font_size=14,
        showlegend=False,
        template="plotly_white",
        margin=dict(t=80, b=40),
    )

    fig.add_vrect(
        x0="2020-03", x1="2021-06",
        fillcolor="gray", opacity=0.07,
        layer="below", line_width=0,
        annotation_text="COVID",
        annotation_position="top left",
        annotation_font_size=9,
    )

    for date_str, label, color in BCE_EVENTS:
        fig.add_vline(
            x=date_str, line_dash="dot", line_color=color, opacity=0.5,
            annotation_text=label, annotation_position="top right",
            annotation_font_size=9, annotation_font_color=color,
        )

    fig.show()

---
## 5. Graphique narratif : cycle BCE 2022-2024

La BCE a relevé ses taux de 0% à 4,5% entre juillet 2022 et septembre 2023.  
Ce graphique montre la transmission sur les indicateurs belges : OLO et EURIBOR  
montent ensemble, l'inflation culmine puis redescend, le chômage reste stable.  
C'est la narrative centrale du dashboard pour les entretiens bancaires.

In [ ]:
if macro.empty:
    print("Pas de données. Relancer les cellules 2 et 3.")
else:
    df_plot = macro["2019":"2026"].copy()

    COLOR_MAP = {
        "olo_10y":    ("OLO 10Y (%)",         "#1a5276", "solid"),
        "euribor_3m": ("EURIBOR 3M (%)",      "#c0392b", "dash"),
        "inflation":  ("Inflation IPCH (%)",  "#d35400", "dot"),
        "chomage":    ("Chomage (% actifs)",   "#6c3483", "dashdot"),
    }

    fig2 = go.Figure()
    for key, (label, color, dash) in COLOR_MAP.items():
        if key not in df_plot.columns:
            continue
        s = df_plot[key].dropna()
        fig2.add_trace(go.Scatter(
            x=s.index, y=s.values,
            mode="lines",
            name=label,
            line=dict(color=color, width=2.2, dash=dash),
            hovertemplate="%{x|%Y-%m}: %{y:.2f}<extra></extra>",
        ))

    for date_str, label, color in BCE_EVENTS:
        fig2.add_vline(
            x=date_str, line_dash="dot", line_color=color, opacity=0.5,
            annotation_text=label, annotation_position="top right",
            annotation_font_size=9, annotation_font_color=color,
        )

    fig2.add_vrect(
        x0="2020-03", x1="2021-06",
        fillcolor="gray", opacity=0.07, layer="below", line_width=0,
        annotation_text="COVID", annotation_position="top left",
        annotation_font_size=9,
    )

    fig2.update_layout(
        title="Cycle BCE 2022-2024 : impact sur les indicateurs belges",
        xaxis_title="Période",
        yaxis_title="Valeur (%)",
        template="plotly_white",
        height=480,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        hovermode="x unified",
    )
    fig2.show()

    try:
        out = DATA_PROCESSED / "narrative_bce_plotly.png"
        fig2.write_image(str(out), width=1200, height=480, scale=1.5)
        print(f"PNG sauvegardé : {out}")
    except Exception as e:
        print(f"Export PNG ignoré (kaleido non installé) : {e}")

---
## 6. Matrice de corrélation

In [ ]:
if macro.empty:
    print("Pas de données. Relancer les cellules 2 et 3.")
else:
    labels_map = {k: v["label"] for k, v in DATASETS.items()}
    cols_available = [c for c in DATASETS if c in macro.columns]

    corr = macro[cols_available].corr(method="pearson").round(3)
    labels_display = [labels_map.get(c, c) for c in corr.columns]

    fig3 = go.Figure(go.Heatmap(
        z=corr.values,
        x=labels_display,
        y=labels_display,
        colorscale="RdYlGn",
        zmin=-1, zmax=1,
        text=corr.values.round(2),
        texttemplate="%{text}",
        textfont_size=12,
        hovertemplate="%{y} x %{x} : %{z:.3f}<extra></extra>",
    ))
    fig3.update_layout(
        title="Matrice de corrélation : indicateurs macro belges (Pearson)",
        template="plotly_white",
        height=420,
        width=560,
        xaxis=dict(tickangle=-30),
    )
    fig3.show()

    print("\nCorrélations fortes (|r| > 0.6) :")
    for i in range(len(corr)):
        for j in range(i + 1, len(corr)):
            r = corr.values[i, j]
            if abs(r) > 0.6:
                print(f"  {corr.index[i]:<12} x {corr.columns[j]:<12} : r = {r:.3f}")

---
## 7. Bilan Jour 1

**Livrables produits :**
- `data/raw/{key}.csv` : 5 séries brutes Eurostat (2010-2026)
- `data/processed/macro_monthly.csv` : DataFrame mensuel consolidé
- `data/processed/narrative_bce_plotly.png` : graphique narratif exporté

**Modules `src/` opérationnels :**
- `config.py` : catalogue des datasets et événements BCE
- `fetch_eurostat.py` : EurostatClient avec cache CSV
- `transform.py` : build_macro_df (fréquences mixtes vers mensuel)

**Narrative bancaire validée :**  
Le cycle BCE 2022-2024 est visible sur les données : OLO et EURIBOR montent ensemble,  
l'inflation culmine, le chômage reste stable. Les corrélations sont mesurées.

**Suite Jour 2 :**
- `src/persist.py` : stocker le DataFrame dans DuckDB
- Ajouter le taux de crédit immobilier (Eurostat `mir_ir_fir`) et les faillites Statbel
- Lancer `streamlit run app/Home.py` et tester l'interface

In [ ]:
raw_files = list(DATA_RAW.glob("*.csv"))
proc_files = list(DATA_PROCESSED.glob("*"))

print("data/raw/ :")
for f in raw_files:
    print(f"  {f.name:<35} {f.stat().st_size:>8} bytes")

print("\ndata/processed/ :")
for f in proc_files:
    print(f"  {f.name:<35} {f.stat().st_size:>8} bytes")